In [7]:
!pip install kagglehub --quiet

import os
os.environ['KAGGLE_USERNAME'] = 'danielfreitas93'
os.environ['KAGGLE_KEY'] = '24abc2f21fd2bbaca2d17503b9e62344'


In [8]:
import kagglehub

path = kagglehub.competition_download('ieee-fraud-detection')
print("Path to competition files:", path)

100%|██████████| 118M/118M [00:04<00:00, 27.2MB/s] 

Extracting files...


Path to competition files: /home/jupyter/.cache/kagglehub/competitions/ieee-fraud-detection


In [9]:
import os
for arquivo in os.listdir(path):
    tamanho_mb = os.path.getsize(os.path.join(path, arquivo)) / (1024 * 1024)
    print(f"{arquivo}: {tamanho_mb:.2f} MB")

test_transaction.csv: 584.79 MB
test_identity.csv: 24.60 MB
train_transaction.csv: 651.69 MB
sample_submission.csv: 5.80 MB
train_identity.csv: 25.30 MB


In [10]:
import pandas as pd

df_train_transaction = pd.read_csv(f'{path}/train_transaction.csv')
df_train_identity = pd.read_csv(f'{path}/train_identity.csv')

print(f'train_transaction: {df_train_transaction.shape[0]:,} linhas, {df_train_transaction.shape[1]} colunas')
print(f'train_identity: {df_train_identity.shape[0]:,} linhas, {df_train_identity.shape[1]} colunas')

train_transaction: 590,540 linhas, 394 colunas
train_identity: 144,233 linhas, 41 colunas


In [12]:
from google.cloud import storage

client = storage.Client()
bucket_name = 'fraud-detection-raw-dfreitas39'  # o bucket que você já criou
bucket = client.bucket(bucket_name)

arquivos_para_subir = [
    'train_transaction.csv',
    'train_identity.csv',
    'test_transaction.csv',
    'test_identity.csv'
]

for arquivo in arquivos_para_subir:
    blob = bucket.blob(f'raw/{arquivo}')
    blob.upload_from_filename(f'{path}/{arquivo}')
    print(f'Enviado: {arquivo}')

Enviado: train_transaction.csv
Enviado: train_identity.csv
Enviado: test_transaction.csv
Enviado: test_identity.csv


In [15]:
from google.cloud import bigquery

bq_client = bigquery.Client()
project_id = 'projeto-fraude-iee-cis'  # confirme o Project ID exato do seu projeto
bucket_name = 'fraud-detection-raw-dfreitas39'

def carregar_csv_para_bronze(nome_arquivo, nome_tabela):
    uri = f'gs://{bucket_name}/raw/{nome_arquivo}'
    tabela_destino = f'{project_id}.fraud_bronze.{nome_tabela}'

    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.CSV,
        skip_leading_rows=1,
        autodetect=True,
        write_disposition='WRITE_TRUNCATE'
    )

    job = bq_client.load_table_from_uri(uri, tabela_destino, job_config=job_config)
    job.result()  # espera o job terminar antes de seguir
    print(f'{nome_tabela} carregada com sucesso')

carregar_csv_para_bronze('train_transaction.csv', 'train_transaction')
carregar_csv_para_bronze('train_identity.csv', 'train_identity')
carregar_csv_para_bronze('test_transaction.csv', 'test_transaction')
carregar_csv_para_bronze('test_identity.csv', 'test_identity')

train_transaction carregada com sucesso
train_identity carregada com sucesso
test_transaction carregada com sucesso
test_identity carregada com sucesso
